# Plan (about 2 weeks)

 - Days 1-2: Create the repo and README. Load the data and check the class balance (how many real vs. fake). Remove duplicates, again before splitting.
 - Days 3-4: Clean the text columns (title, description, requirements) and handle missing values. Combine them into one text field. EDA: what do fake postings have in common (missing company info, no requirements, very short text)?
 - Days 5-7: Split with stratify=y, so both sets keep the same fake/real ratio. Build a pipeline with TF-IDF and Logistic Regression as your baseline.
 - Days 8-10: Try a second model (Linear SVM or XGBoost on TF-IDF) and handle the imbalance with class_weight="balanced". Tune the decision threshold.
 - Days 11-12: Look at the most important words, and read some of the errors the model makes. Build a Streamlit app where you paste a job ad    and get a risk score.
 - Days 13-14: Deploy, write the README, and post on LinkedIn.

In [136]:
import pandas as pd
import numpy as np
import seaborn as sns

from sklearn.model_selection import train_test_split

In [110]:
TARGET_COL = 'fraudulent'
RANDOM_STATE = 42

In [111]:
df = pd.read_csv("fake_job_postings.csv")

In [112]:
df.head()

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


In [113]:
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nMissing values per column:")
print(df.isnull().sum().sort_values(ascending=False))
print("\nDuplicate rows:", df.duplicated().sum())

Shape: (17880, 18)

Dtypes:
 job_id                  int64
title                  object
location               object
department             object
salary_range           object
company_profile        object
description            object
requirements           object
benefits               object
telecommuting           int64
has_company_logo        int64
has_questions           int64
employment_type        object
required_experience    object
required_education     object
industry               object
function               object
fraudulent              int64
dtype: object

Missing values per column:
salary_range           15012
department             11547
required_education      8105
benefits                7212
required_experience     7050
function                6455
industry                4903
employment_type         3471
company_profile         3308
requirements            2696
location                 346
description                1
job_id                     0
telecommutin

In [114]:
df.head(3)

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0


In [115]:
df_columns = [col for col in df.columns.tolist() if col != 'job_id']
df.duplicated(subset=df_columns).sum()

np.int64(281)

In [116]:
print(df[TARGET_COL].value_counts())
print("-"*20)
print(df[TARGET_COL].value_counts(normalize=True))

fraudulent
0    17014
1      866
Name: count, dtype: int64
--------------------
fraudulent
0    0.951566
1    0.048434
Name: proportion, dtype: float64


In [117]:
dup_columns = [c for c in df.columns if c not in ("job_id", TARGET_COL)]
print(dup_columns)
print("-"*35)
df.duplicated(subset=dup_columns).sum()

['title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function']
-----------------------------------


np.int64(281)

In [118]:
df[df.duplicated(subset=dup_columns)][TARGET_COL].value_counts()

fraudulent
0    271
1     10
Name: count, dtype: int64

In [119]:
df = df.drop_duplicates(subset=dup_columns)

In [120]:
df.duplicated(subset=dup_columns).sum()

np.int64(0)

In [121]:
df[TARGET_COL].value_counts()

fraudulent
0    16743
1      856
Name: count, dtype: int64

In [122]:
df[df['description'].isnull()]

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
17513,17514,Office Manager,"PL, MZ, Warsaw",NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,NaN,NaN,NaN,NaN,NaN,1


In [123]:
missing_columns = ["salary_range", "company_profile", "benefits" , "department"]

for col in missing_columns:
    df[f"{col}_missing"] = df[col].isna().astype(int)

In [124]:
df.head()

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,...,employment_type,required_experience,required_education,industry,function,fraudulent,salary_range_missing,company_profile_missing,benefits_missing,department_missing
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,...,Other,Internship,NaN,NaN,Marketing,0,1,0,1,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,...,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0,1,0,0,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,...,NaN,NaN,NaN,NaN,NaN,0,1,0,1,1
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,...,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0,1,0,0,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,...,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0,1,0,0,1


In [125]:
flag_cols = [c + "_missing" for c in missing_columns]
print(df[flag_cols].mean())
print(df.shape)

salary_range_missing       0.839366
company_profile_missing    0.186488
benefits_missing           0.403887
department_missing         0.643673
dtype: float64
(17599, 22)


In [126]:
df[TARGET_COL].value_counts()

fraudulent
0    16743
1      856
Name: count, dtype: int64

In [132]:
X = df.drop(columns=[TARGET_COL, "job_id"])
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    random_state=RANDOM_STATE,
    stratify=y,
    test_size=0.2
)

print(y_train.value_counts(normalize=True))
print("-"*50)
print(y_test.value_counts(normalize=True))
print("-"*50)
print(X_train.shape, X_test.shape)

fraudulent
0    0.951346
1    0.048654
Name: proportion, dtype: float64
--------------------------------------------------
fraudulent
0    0.95142
1    0.04858
Name: proportion, dtype: float64
--------------------------------------------------
(14079, 20) (3520, 20)


In [134]:
for col in flag_cols:
    print(y_train.groupby(X_train[col]).mean())
    print("-" * 30)

salary_range_missing
0    0.078912
1    0.042920
Name: fraudulent, dtype: float64
------------------------------
company_profile_missing
0    0.019850
1    0.176244
Name: fraudulent, dtype: float64
------------------------------
benefits_missing
0    0.047676
1    0.050106
Name: fraudulent, dtype: float64
------------------------------
department_missing
0    0.053543
1    0.045941
Name: fraudulent, dtype: float64
------------------------------


In [135]:
((X_train["company_profile_missing"] == 1) & (y_train == 1)).sum()

np.int64(457)